In [1]:
from pxd.datasets.hdf5.directory import HDF5DirectoryDataset, HDF5DirectoryDatasetConfig
import numpy as np
import pandas as pd
from typing import Dict, Any, Tuple, List
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from samples.sample import Sample

In [2]:
def get_samples(data_path):
    config = HDF5DirectoryDatasetConfig(
        path=data_path
    )

    dataset = HDF5DirectoryDataset(config)

    dataset_dict = dataset.load()

    samples = {}
    for i in tqdm(list(dataset_dict.keys())):
        samples[i] = dataset_dict[i]()
    
    return samples

In [3]:
train_pred_path="/flux/vault/stator/04_modelling/ds3_polynomial_ridge/train_predictions"
test_pred_path="/flux/vault/stator/04_modelling/ds3_polynomial_ridge/test_predictions"
val_pred_path="/flux/vault/stator/04_modelling/ds3_polynomial_ridge/validation_predictions"

In [3]:
test_pred_path="/flux/vault/stator/04_modelling/ds3_polynomial_ridge/test_predictions"
test_split_path = "flux/vault/stator/03_splitting/stator_splitting_ds3_jul13/test/"

In [4]:
train_samples = get_samples(train_pred_path)

100%|██████████| 800/800 [00:07<00:00, 103.48it/s]


In [5]:
test_samples = get_samples(test_pred_path)

100%|██████████| 100/100 [00:00<00:00, 102.48it/s]


In [6]:
val_samples = get_samples(val_pred_path)

100%|██████████| 100/100 [00:00<00:00, 101.26it/s]


# Extract and Stack `heatflux_matrix` Data

This notebook extracts the following keys from each sample's `data -> heatflux_matrix` dictionary:

- `geometry_parameter_a_matrix`
- `a_matrix_coeffiecient_positions`
- `a_matrix_coefficients`

For each sample, the arrays are normalized to shapes `(120, 20)`, `(120, 2)`, and `(120, 1)`, then horizontally stacked into `(120, 23)`.
All samples are then vertically stacked into a final dataset of shape `(120 * number_of_samples, 23)`.

In [ ]:

# ------------------------------
# Helper functions
# ------------------------------

def _ensure_2d(arr: Any) -> np.ndarray:
    """Convert input to a 2D numpy array."""
    arr = np.asarray(arr)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D array, got shape {arr.shape}")
    return arr


def _normalize_shape(arr: Any, expected_cols: int, rows_per_sample: int = 120, name: str = 'array') -> np.ndarray:
    """
    Normalize an input matrix so its final shape is (rows_per_sample, expected_cols).

    Accepts either:
      - (rows_per_sample, expected_cols)
      - (expected_cols, rows_per_sample)  -> transposed automaticallyEx
    """
    arr = _ensure_2d(arr)

    if arr.shape == (rows_per_sample, expected_cols):
        return arr
    elif arr.shape == (expected_cols, rows_per_sample):
        return arr.T
    else:
        raise ValueError(
            f"Unexpected shape for {name}: {arr.shape}. "
            f"Expected ({rows_per_sample}, {expected_cols}) or ({expected_cols}, {rows_per_sample})."
        )


def np_to_df(full_dataset):
    feature_names = [f"geometry_parameter_a_{i+1}" for i in range(20)] + [
        "a_matrix_coeff_position_1",
        "a_matrix_coeff_position_2",
        "a_matrix_coefficient",
        "a_matrix_coefficients_pred",
        "a_matrix_coefficients_pred_stddev",
        
    ]

    df = pd.DataFrame(full_dataset, columns=feature_names)

    return df

# ------------------------------
# Main extraction function
# ------------------------------

def build_heatflux_dataset(
    samples_dict: Dict[str, Dict[str, Any]],
    rows_per_sample: int = 120
) -> Tuple[np.ndarray, Dict[str, np.ndarray], List[str]]:
    """
    Extracts and combines data from each sample:
      - geometry_parameter_a_matrix           -> 20 columns
      - a_matrix_coeffiecient_positions       -> 2 columns
      - a_matrix_coefficients                 -> 1 column

    For each sample, the three arrays are horizontally stacked into shape:
        (rows_per_sample, 23)

    Then all sample matrices are vertically stacked into shape:
        (rows_per_sample * number_of_samples, 23)

    Parameters
    ----------
    samples_dict : dict
        Expected structure:
        {
            sample_id: {
                "data": {
                    "heatflux_matrix": {
                        "geometry_parameter_a_matrix": ...,
                        "a_matrix_coeffiecient_positions": ...,
                        "a_matrix_coefficients": ...
                    },
                    ... # other dictionaries ignored
                }
            },
            ...
        }

    rows_per_sample : int, default=120
        Number of rows for each sample.

    Returns
    -------
    full_dataset : np.ndarray
        Shape = (rows_per_sample * n_samples, 23)

    per_sample_data : dict
        {sample_id: sample_matrix} where each sample_matrix has shape (rows_per_sample, 23)

    sample_order : list
        Order in which samples were processed and stacked.
    """
    per_sample_data: Dict[str, np.ndarray] = {}
    sample_order: List[str] = []

    for sample_id, sample_payload in samples_dict.items():
        try:
            heatflux = sample_payload["data"]["heatflux_matrix"]
        except KeyError as e:
            raise KeyError(f"Missing key for sample '{sample_id}': {e}") from e

        geom_a = _normalize_shape(
            heatflux["geometry_parameter_a_matrix"],
            expected_cols=20,
            rows_per_sample=rows_per_sample,
            name=f"{sample_id}.geometry_parameter_a_matrix"
        )

  

        coeff_pos = _normalize_shape(
            heatflux["a_matrix_coeffiecient_positions"],
            expected_cols=2,
            rows_per_sample=rows_per_sample,
            name=f"{sample_id}.a_matrix_coeffiecient_positions"
        )

        coeffs = _normalize_shape(
            heatflux["a_matrix_coefficients"],
            expected_cols=1,
            rows_per_sample=rows_per_sample,
            name=f"{sample_id}.a_matrix_coefficients"
        )

        coeffs_pred = _normalize_shape(
            heatflux["a_matrix_coefficients_pred"],
            expected_cols=1,
            rows_per_sample=rows_per_sample,
            name=f"{sample_id}.a_matrix_coefficients_pred"
        )

        coeffs_pred_stddev = _normalize_shape(
            heatflux["a_matrix_coefficients_pred_stddev"],
            expected_cols=1,
            rows_per_sample=rows_per_sample,
            name=f"{sample_id}.a_matrix_coefficients_pred_stddev"
        )

        # Horizontal stack within one sample -> (120, 25)
        sample_matrix = np.hstack([geom_a, coeff_pos, coeffs, coeffs_pred, coeffs_pred_stddev])

        if sample_matrix.shape != (rows_per_sample, 25):
            raise ValueError(
                f"Unexpected combined shape for sample '{sample_id}': {sample_matrix.shape}. "
                f"Expected ({rows_per_sample}, 25)."
            )

        per_sample_data[sample_id] = sample_matrix
        sample_order.append(sample_id)

    # Vertical stack across samples -> (120 * n_samples, 25)
    if not per_sample_data:
        full_dataset = np.empty((0, 25))
    else:
        full_dataset = np.vstack([per_sample_data[sid] for sid in sample_order])

    full_df = np_to_df(full_dataset)

    return full_df, per_sample_data, sample_order


In [11]:
train_df, _, train_sample_order = build_heatflux_dataset(train_samples)

print("Sample order:", train_sample_order)
'''for sid, arr in per_sample_data.items():
    print(f"{sid}: {arr.shape}")'''

train_df.head() # Expected: (120 * number_of_samples, 25)

Sample order: ['sample_207', 'sample_258', 'sample_156', 'sample_392', 'sample_661', 'sample_598', 'sample_703', 'sample_54', 'sample_696', 'sample_820', 'sample_530', 'sample_611', 'sample_773', 'sample_784', 'sample_850', 'sample_540', 'sample_422', 'sample_277', 'sample_315', 'sample_228', 'sample_126', 'sample_280', 'sample_179', 'sample_557', 'sample_435', 'sample_818', 'sample_508', 'sample_925', 'sample_847', 'sample_793', 'sample_606', 'sample_33', 'sample_764', 'sample_659', 'sample_297', 'sample_131', 'sample_385', 'sample_141', 'sample_210', 'sample_372', 'sample_445', 'sample_868', 'sample_681', 'sample_578', 'sample_955', 'sample_837', 'sample_676', 'sample_1000', 'sample_373', 'sample_211', 'sample_384', 'sample_628', 'sample_715', 'sample_42', 'sample_677', 'sample_836', 'sample_954', 'sample_579', 'sample_680', 'sample_869', 'sample_444', 'sample_526', 'sample_32', 'sample_765', 'sample_607', 'sample_792', 'sample_846', 'sample_509', 'sample_819', 'sample_434', 'sample_

,geometry_parameter_a_1,geometry_parameter_a_2,geometry_parameter_a_3,geometry_parameter_a_4,geometry_parameter_a_5,geometry_parameter_a_6,geometry_parameter_a_7,geometry_parameter_a_8,geometry_parameter_a_9,geometry_parameter_a_10,...,geometry_parameter_a_16,geometry_parameter_a_17,geometry_parameter_a_18,geometry_parameter_a_19,geometry_parameter_a_20,a_matrix_coeff_position_1,a_matrix_coeff_position_2,a_matrix_coefficient,a_matrix_coefficients_pred,a_matrix_coefficients_pred_stddev
0,1.473808,1.164271,0.411383,0.188484,1.447723,2.244252,2.763662,0.963613,2.33142,0.293598,...,0.668821,1.249661,0.268543,0.07897,0.006194,0.0,1.0,0.041215,0.223282,0.459047
1,1.473808,1.164271,0.411383,0.188484,1.447723,2.244252,2.763662,0.963613,2.33142,0.293598,...,0.668821,1.249661,0.268543,0.07897,0.006194,0.0,2.0,23.658109,23.630709,0.195497
2,1.473808,1.164271,0.411383,0.188484,1.447723,2.244252,2.763662,0.963613,2.33142,0.293598,...,0.668821,1.249661,0.268543,0.07897,0.006194,0.0,3.0,46.539495,47.260829,2.164117
3,1.473808,1.164271,0.411383,0.188484,1.447723,2.244252,2.763662,0.963613,2.33142,0.293598,...,0.668821,1.249661,0.268543,0.07897,0.006194,0.0,4.0,0.540085,0.540427,0.003131
4,1.473808,1.164271,0.411383,0.188484,1.447723,2.244252,2.763662,0.963613,2.33142,0.293598,...,0.668821,1.249661,0.268543,0.07897,0.006194,0.0,5.0,0.000647,0.000647,0.000073


In [12]:
test_df, _, test_sample_order = build_heatflux_dataset(test_samples)

In [13]:
val_df, _, val_sample_order = build_heatflux_dataset(val_samples)

In [14]:
train_sample_order = np.repeat(train_sample_order, 120)
test_sample_order = np.repeat(test_sample_order, 120)
val_sample_order = np.repeat(val_sample_order, 120)

In [15]:
test_df.index = test_sample_order
val_df.index = val_sample_order
train_df.index = train_sample_order

In [16]:
# Save as CSV if needed
train_df.to_csv(r"data/full_train_prediction_botorch_dataset.csv", index=True)

print("Saved: full_heatflux_dataset.npy and full_heatflux_dataset.csv")

Saved: full_heatflux_dataset.npy and full_heatflux_dataset.csv


In [17]:
test_df.to_csv(r"data/full_test_prediction_botorch_dataset.csv", index=True)
val_df.to_csv(r"data/full_val_prediction_botorch_dataset.csv", index=True)

## Build and Save Datasaet for 